# Análise exploratória | ME3N | tbl_ds_pro_me3n_contratos_compra

| Propriedade | Valor |
|---|---|
| Tabela | ``dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`` |
| Ambiente | DEV |
| Objeto | Tabela |
| Colunas do DESCRIBE | 48 |
| Tipos | bigint, boolean, date, decimal(13,3), decimal(5,0), double, string |
| Clustering declarado | `num_documento_compras`, `cod_material`, `cod_centro`, `dt_fim_validade` |
| Centro | `cod_centro` |
| Chave candidata | `num_documento_compras` + `num_item_documento_compras` (validar na seção 4) |

**Como usar:** importar no Databricks e executar Run All. Células SQL independentes, somente leitura. Resultados são medições, não conclusões prévias. Para views, conferir no DESCRIBE EXTENDED se filtros da tela SAP são aplicados. Não executar cenários antes de analisar os resultados. Fontes: `DESCRIBE_ME3N.csv` e `01_PADROES_SQL_NOTEBOOKS.md`.

### Alertas do schema
| Severidade | Alerta |
|---|---|
| 🔴 ALTA | 1 campos tratados como potencialmente pessoais; amostras só com hash ou omitidos. |
| 🟡 MÉDIA | Valores DOUBLE: conferir precisão e tolerância de 0,005 na comparação. |

### Índice
1 Metadados, 2 Volumetria, 3 Distribuição por centro, 4 Granularidade, 5 Duplicidade, 6 Preenchimento de todas as colunas, 6.1 Colunas sem fonte, 7 Cardinalidade, 8 Domínio, 9 Booleanas e indicadores, 10 Perfil numérico, 11 Totais para conciliação, 12 Datas, 13 Códigos, 14 Checksum, 15 Amostra, 16 Distribuição interna, 17 Freshness, 18 Específicas da transação, 19 Amostra ampliada, 20 Resumo. Inclui subseções 3.1 e 6.1.


## 1. Metadados

Inspeciona propriedades e comentário do objeto para conferir escopo e eventuais filtros da transação.


In [0]:
DESCRIBE EXTENDED `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`;


### 1.1. Detalhes e histórico da tabela

Comandos independentes, executados somente em tabelas.


In [0]:
DESCRIBE DETAIL `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`;


In [0]:
DESCRIBE HISTORY `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`;


## 2. Volumetria

Conta registros e entidades existentes, sem supor que campos de identificação formam uma chave.


In [0]:
SELECT COUNT(*) AS linhas, COUNT(DISTINCT `cod_fornecedor`) AS distintos_cod_fornecedor, COUNT(DISTINCT `num_documento_compras`) AS distintos_num_documento_compras, COUNT(DISTINCT `cod_material`) AS distintos_cod_material, COUNT(DISTINCT `num_item_documento_compras`) AS distintos_num_item_documento_compras, COUNT(DISTINCT `cod_centro`) AS centros_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`;


## 3. Distribuição por centro

Mede volumes e sinaliza centros entre 10 mil e 300 mil registros, sem criar cenários.


In [0]:
SELECT `cod_centro` AS centro, COUNT(*) AS linhas, CASE WHEN COUNT(*) BETWEEN 10000 AND 300000 THEN 'CANDIDATO A CENARIO' ELSE 'AVALIAR VOLUME' END AS avaliacao FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `cod_centro` ORDER BY linhas DESC;


## 3.1. Centros 4128 e 4014

Confirma a presença e o formato literal dos centros padrão; ainda não filtra os demais testes.


In [0]:
SELECT `cod_centro` AS centro_literal, LENGTH(TRIM(CAST(`cod_centro` AS STRING))) AS tamanho, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` WHERE TRIM(CAST(`cod_centro` AS STRING)) RLIKE '^0*(4128|4014)$' GROUP BY `cod_centro`, LENGTH(TRIM(CAST(`cod_centro` AS STRING))) ORDER BY centro_literal;


## 4. Granularidade

Compara 3 a 5 chaves candidatas. Uma razão igual a 1 é indício, não prova isolada de chave de negócio.


In [0]:
WITH d AS (SELECT 'num_documento_compras' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento_compras` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) k UNION ALL SELECT 'num_documento_compras + num_item_documento_compras' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento_compras`, `num_item_documento_compras` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) k UNION ALL SELECT 'num_documento_compras + num_item_documento_compras + cod_centro' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento_compras`, `num_item_documento_compras`, `cod_centro` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) k UNION ALL SELECT 'num_documento_compras + num_item_documento_compras + cod_material' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento_compras`, `num_item_documento_compras`, `cod_material` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) k), v AS (SELECT COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) SELECT d.chave_candidata, v.linhas, d.chaves_distintas, ROUND(v.linhas / NULLIF(d.chaves_distintas, 0), 4) AS linhas_por_chave FROM d CROSS JOIN v ORDER BY linhas_por_chave, chave_candidata;


## 5. Duplicidade

Separa repetições de chave entre linhas idênticas e versões diferentes, com hash da linha inteira sem exibir dados pessoais.


In [0]:
WITH h AS (SELECT t.`num_documento_compras`, t.`num_item_documento_compras`, md5(to_json(struct(t.`num_documento_compras`, t.`num_item_documento_compras`, t.`tp_categoria_documento_compras`, t.`tp_documento_compras`, t.`cod_material`, t.`desc_texto_breve_item`, t.`cod_centro`, t.`cod_deposito`, t.`cod_grupo_mercadorias`, t.`cod_unidade_medida_pedido`, t.`cod_unidade_medida_basica`, t.`cod_unidade_medida_preco_pedido`, t.`vl_preco_liquido`, t.`qt_unidade_preco`, t.`qt_prevista_contrato`, t.`vl_fixado_item`, t.`cod_iva`, t.`cod_domicilio_fiscal`, t.`cod_registro_info_compras`, t.`tp_categoria_item_compra`, t.`tp_classificacao_contabil`, t.`cod_eliminacao_item`, t.`cod_empresa`, t.`cod_moeda`, t.`cod_fornecedor`, t.`nm_fornecedor`, t.`cod_grupo_compradores`, t.`cod_organizacao_compras`, t.`dt_inicio_validade`, t.`dt_fim_validade`, t.`dt_documento_compras`, t.`dt_criacao_cabecalho`, t.`cod_liberacao`, t.`cod_eliminacao_cabecalho`, t.`vl_fixado_cabecalho`, t.`cod_centro_custo`, t.`num_ordem`, t.`id_interno_elemento_pep`, t.`qtd_classificacoes_contabeis`, t.`vlr_consumido`, t.`qtd_itens_pedido_consumo`, t.`ind_servico`, t.`ind_possui_cabecalho`, t.`ind_contrato_multicentro`, t.`ind_possui_consumo`, t.`dateingest`, t.`yearingest`, t.`monthingest`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` t), d AS (SELECT `num_documento_compras`, `num_item_documento_compras`, COUNT(*) AS linhas, COUNT(DISTINCT hash_linha) AS versoes FROM h GROUP BY `num_documento_compras`, `num_item_documento_compras` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COALESCE(SUM(linhas),0) AS linhas_em_chaves_repetidas, COALESCE(SUM(linhas - versoes),0) AS repeticoes_identicas, COALESCE(SUM(CASE WHEN versoes > 1 THEN 1 ELSE 0 END),0) AS chaves_com_versoes_diferentes FROM d;


## 5.1. Atributos variáveis nas chaves repetidas

Mede variação de atributos não pessoais nas chaves candidatas repetidas; ausência de variação não prova identidade da linha completa.


In [0]:
WITH d AS (SELECT `num_documento_compras`, `num_item_documento_compras`, COUNT(*) AS linhas, COUNT(DISTINCT CAST(`num_documento_compras` AS STRING)) AS `dist_num_documento_compras`, COUNT(DISTINCT CAST(`num_item_documento_compras` AS STRING)) AS `dist_num_item_documento_compras`, COUNT(DISTINCT CAST(`tp_categoria_documento_compras` AS STRING)) AS `dist_tp_categoria_documento_compras`, COUNT(DISTINCT CAST(`tp_documento_compras` AS STRING)) AS `dist_tp_documento_compras`, COUNT(DISTINCT CAST(`cod_material` AS STRING)) AS `dist_cod_material`, COUNT(DISTINCT CAST(`desc_texto_breve_item` AS STRING)) AS `dist_desc_texto_breve_item`, COUNT(DISTINCT CAST(`cod_centro` AS STRING)) AS `dist_cod_centro`, COUNT(DISTINCT CAST(`cod_deposito` AS STRING)) AS `dist_cod_deposito`, COUNT(DISTINCT CAST(`cod_grupo_mercadorias` AS STRING)) AS `dist_cod_grupo_mercadorias`, COUNT(DISTINCT CAST(`cod_unidade_medida_pedido` AS STRING)) AS `dist_cod_unidade_medida_pedido`, COUNT(DISTINCT CAST(`cod_unidade_medida_basica` AS STRING)) AS `dist_cod_unidade_medida_basica`, COUNT(DISTINCT CAST(`cod_unidade_medida_preco_pedido` AS STRING)) AS `dist_cod_unidade_medida_preco_pedido` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `num_documento_compras`, `num_item_documento_compras` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COUNT_IF(`dist_num_documento_compras` > 1) AS `chaves_variam_num_documento_compras`, COUNT_IF(`dist_num_item_documento_compras` > 1) AS `chaves_variam_num_item_documento_compras`, COUNT_IF(`dist_tp_categoria_documento_compras` > 1) AS `chaves_variam_tp_categoria_documento_compras`, COUNT_IF(`dist_tp_documento_compras` > 1) AS `chaves_variam_tp_documento_compras`, COUNT_IF(`dist_cod_material` > 1) AS `chaves_variam_cod_material`, COUNT_IF(`dist_desc_texto_breve_item` > 1) AS `chaves_variam_desc_texto_breve_item`, COUNT_IF(`dist_cod_centro` > 1) AS `chaves_variam_cod_centro`, COUNT_IF(`dist_cod_deposito` > 1) AS `chaves_variam_cod_deposito`, COUNT_IF(`dist_cod_grupo_mercadorias` > 1) AS `chaves_variam_cod_grupo_mercadorias`, COUNT_IF(`dist_cod_unidade_medida_pedido` > 1) AS `chaves_variam_cod_unidade_medida_pedido`, COUNT_IF(`dist_cod_unidade_medida_basica` > 1) AS `chaves_variam_cod_unidade_medida_basica`, COUNT_IF(`dist_cod_unidade_medida_preco_pedido` > 1) AS `chaves_variam_cod_unidade_medida_preco_pedido` FROM d;


## 6. Preenchimento de todas as colunas

Mede nulos, vazios textuais, zeros e úteis em TODAS as colunas. Zero em código pode ser valor legítimo; veredito não equivale a defeito.


In [0]:
WITH perf AS (SELECT stack(24, 'num_documento_compras', 'string', c0_n, c0_v, c0_z, 'num_item_documento_compras', 'string', c1_n, c1_v, c1_z, 'tp_categoria_documento_compras', 'string', c2_n, c2_v, c2_z, 'tp_documento_compras', 'string', c3_n, c3_v, c3_z, 'cod_material', 'string', c4_n, c4_v, c4_z, 'desc_texto_breve_item', 'string', c5_n, c5_v, c5_z, 'cod_centro', 'string', c6_n, c6_v, c6_z, 'cod_deposito', 'string', c7_n, c7_v, c7_z, 'cod_grupo_mercadorias', 'string', c8_n, c8_v, c8_z, 'cod_unidade_medida_pedido', 'string', c9_n, c9_v, c9_z, 'cod_unidade_medida_basica', 'string', c10_n, c10_v, c10_z, 'cod_unidade_medida_preco_pedido', 'string', c11_n, c11_v, c11_z, 'vl_preco_liquido', 'double', c12_n, c12_v, c12_z, 'qt_unidade_preco', 'decimal(5,0)', c13_n, c13_v, c13_z, 'qt_prevista_contrato', 'decimal(13,3)', c14_n, c14_v, c14_z, 'vl_fixado_item', 'double', c15_n, c15_v, c15_z, 'cod_iva', 'string', c16_n, c16_v, c16_z, 'cod_domicilio_fiscal', 'string', c17_n, c17_v, c17_z, 'cod_registro_info_compras', 'string', c18_n, c18_v, c18_z, 'tp_categoria_item_compra', 'string', c19_n, c19_v, c19_z, 'tp_classificacao_contabil', 'string', c20_n, c20_v, c20_z, 'cod_eliminacao_item', 'string', c21_n, c21_v, c21_z, 'cod_empresa', 'string', c22_n, c22_v, c22_z, 'cod_moeda', 'string', c23_n, c23_v, c23_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`num_documento_compras` IS NULL) AS c0_n, COUNT_IF(`num_documento_compras` IS NOT NULL AND LOWER(TRIM(`num_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c0_v, COUNT_IF(TRIM(`num_documento_compras`) RLIKE '^0+([.,]0+)?$') AS c0_z, COUNT_IF(`num_item_documento_compras` IS NULL) AS c1_n, COUNT_IF(`num_item_documento_compras` IS NOT NULL AND LOWER(TRIM(`num_item_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c1_v, COUNT_IF(TRIM(`num_item_documento_compras`) RLIKE '^0+([.,]0+)?$') AS c1_z, COUNT_IF(`tp_categoria_documento_compras` IS NULL) AS c2_n, COUNT_IF(`tp_categoria_documento_compras` IS NOT NULL AND LOWER(TRIM(`tp_categoria_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c2_v, COUNT_IF(TRIM(`tp_categoria_documento_compras`) RLIKE '^0+([.,]0+)?$') AS c2_z, COUNT_IF(`tp_documento_compras` IS NULL) AS c3_n, COUNT_IF(`tp_documento_compras` IS NOT NULL AND LOWER(TRIM(`tp_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c3_v, COUNT_IF(TRIM(`tp_documento_compras`) RLIKE '^0+([.,]0+)?$') AS c3_z, COUNT_IF(`cod_material` IS NULL) AS c4_n, COUNT_IF(`cod_material` IS NOT NULL AND LOWER(TRIM(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c4_v, COUNT_IF(TRIM(`cod_material`) RLIKE '^0+([.,]0+)?$') AS c4_z, COUNT_IF(`desc_texto_breve_item` IS NULL) AS c5_n, COUNT_IF(`desc_texto_breve_item` IS NOT NULL AND LOWER(TRIM(`desc_texto_breve_item`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c5_v, COUNT_IF(TRIM(`desc_texto_breve_item`) RLIKE '^0+([.,]0+)?$') AS c5_z, COUNT_IF(`cod_centro` IS NULL) AS c6_n, COUNT_IF(`cod_centro` IS NOT NULL AND LOWER(TRIM(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c6_v, COUNT_IF(TRIM(`cod_centro`) RLIKE '^0+([.,]0+)?$') AS c6_z, COUNT_IF(`cod_deposito` IS NULL) AS c7_n, COUNT_IF(`cod_deposito` IS NOT NULL AND LOWER(TRIM(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c7_v, COUNT_IF(TRIM(`cod_deposito`) RLIKE '^0+([.,]0+)?$') AS c7_z, COUNT_IF(`cod_grupo_mercadorias` IS NULL) AS c8_n, COUNT_IF(`cod_grupo_mercadorias` IS NOT NULL AND LOWER(TRIM(`cod_grupo_mercadorias`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c8_v, COUNT_IF(TRIM(`cod_grupo_mercadorias`) RLIKE '^0+([.,]0+)?$') AS c8_z, COUNT_IF(`cod_unidade_medida_pedido` IS NULL) AS c9_n, COUNT_IF(`cod_unidade_medida_pedido` IS NOT NULL AND LOWER(TRIM(`cod_unidade_medida_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c9_v, COUNT_IF(TRIM(`cod_unidade_medida_pedido`) RLIKE '^0+([.,]0+)?$') AS c9_z, COUNT_IF(`cod_unidade_medida_basica` IS NULL) AS c10_n, COUNT_IF(`cod_unidade_medida_basica` IS NOT NULL AND LOWER(TRIM(`cod_unidade_medida_basica`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c10_v, COUNT_IF(TRIM(`cod_unidade_medida_basica`) RLIKE '^0+([.,]0+)?$') AS c10_z, COUNT_IF(`cod_unidade_medida_preco_pedido` IS NULL) AS c11_n, COUNT_IF(`cod_unidade_medida_preco_pedido` IS NOT NULL AND LOWER(TRIM(`cod_unidade_medida_preco_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c11_v, COUNT_IF(TRIM(`cod_unidade_medida_preco_pedido`) RLIKE '^0+([.,]0+)?$') AS c11_z, COUNT_IF(`vl_preco_liquido` IS NULL) AS c12_n, CAST(0 AS BIGINT) AS c12_v, COUNT_IF(`vl_preco_liquido` = 0) AS c12_z, COUNT_IF(`qt_unidade_preco` IS NULL) AS c13_n, CAST(0 AS BIGINT) AS c13_v, COUNT_IF(`qt_unidade_preco` = 0) AS c13_z, COUNT_IF(`qt_prevista_contrato` IS NULL) AS c14_n, CAST(0 AS BIGINT) AS c14_v, COUNT_IF(`qt_prevista_contrato` = 0) AS c14_z, COUNT_IF(`vl_fixado_item` IS NULL) AS c15_n, CAST(0 AS BIGINT) AS c15_v, COUNT_IF(`vl_fixado_item` = 0) AS c15_z, COUNT_IF(`cod_iva` IS NULL) AS c16_n, COUNT_IF(`cod_iva` IS NOT NULL AND LOWER(TRIM(`cod_iva`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c16_v, COUNT_IF(TRIM(`cod_iva`) RLIKE '^0+([.,]0+)?$') AS c16_z, COUNT_IF(`cod_domicilio_fiscal` IS NULL) AS c17_n, COUNT_IF(`cod_domicilio_fiscal` IS NOT NULL AND LOWER(TRIM(`cod_domicilio_fiscal`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c17_v, COUNT_IF(TRIM(`cod_domicilio_fiscal`) RLIKE '^0+([.,]0+)?$') AS c17_z, COUNT_IF(`cod_registro_info_compras` IS NULL) AS c18_n, COUNT_IF(`cod_registro_info_compras` IS NOT NULL AND LOWER(TRIM(`cod_registro_info_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c18_v, COUNT_IF(TRIM(`cod_registro_info_compras`) RLIKE '^0+([.,]0+)?$') AS c18_z, COUNT_IF(`tp_categoria_item_compra` IS NULL) AS c19_n, COUNT_IF(`tp_categoria_item_compra` IS NOT NULL AND LOWER(TRIM(`tp_categoria_item_compra`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c19_v, COUNT_IF(TRIM(`tp_categoria_item_compra`) RLIKE '^0+([.,]0+)?$') AS c19_z, COUNT_IF(`tp_classificacao_contabil` IS NULL) AS c20_n, COUNT_IF(`tp_classificacao_contabil` IS NOT NULL AND LOWER(TRIM(`tp_classificacao_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c20_v, COUNT_IF(TRIM(`tp_classificacao_contabil`) RLIKE '^0+([.,]0+)?$') AS c20_z, COUNT_IF(`cod_eliminacao_item` IS NULL) AS c21_n, COUNT_IF(`cod_eliminacao_item` IS NOT NULL AND LOWER(TRIM(`cod_eliminacao_item`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c21_v, COUNT_IF(TRIM(`cod_eliminacao_item`) RLIKE '^0+([.,]0+)?$') AS c21_z, COUNT_IF(`cod_empresa` IS NULL) AS c22_n, COUNT_IF(`cod_empresa` IS NOT NULL AND LOWER(TRIM(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c22_v, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+([.,]0+)?$') AS c22_z, COUNT_IF(`cod_moeda` IS NULL) AS c23_n, COUNT_IF(`cod_moeda` IS NOT NULL AND LOWER(TRIM(`cod_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c23_v, COUNT_IF(TRIM(`cod_moeda`) RLIKE '^0+([.,]0+)?$') AS c23_z FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) a UNION ALL SELECT stack(24, 'cod_fornecedor', 'string', c24_n, c24_v, c24_z, 'nm_fornecedor', 'string', c25_n, c25_v, c25_z, 'cod_grupo_compradores', 'string', c26_n, c26_v, c26_z, 'cod_organizacao_compras', 'string', c27_n, c27_v, c27_z, 'dt_inicio_validade', 'date', c28_n, c28_v, c28_z, 'dt_fim_validade', 'date', c29_n, c29_v, c29_z, 'dt_documento_compras', 'date', c30_n, c30_v, c30_z, 'dt_criacao_cabecalho', 'date', c31_n, c31_v, c31_z, 'cod_liberacao', 'string', c32_n, c32_v, c32_z, 'cod_eliminacao_cabecalho', 'string', c33_n, c33_v, c33_z, 'vl_fixado_cabecalho', 'double', c34_n, c34_v, c34_z, 'cod_centro_custo', 'string', c35_n, c35_v, c35_z, 'num_ordem', 'string', c36_n, c36_v, c36_z, 'id_interno_elemento_pep', 'string', c37_n, c37_v, c37_z, 'qtd_classificacoes_contabeis', 'bigint', c38_n, c38_v, c38_z, 'vlr_consumido', 'double', c39_n, c39_v, c39_z, 'qtd_itens_pedido_consumo', 'bigint', c40_n, c40_v, c40_z, 'ind_servico', 'boolean', c41_n, c41_v, c41_z, 'ind_possui_cabecalho', 'boolean', c42_n, c42_v, c42_z, 'ind_contrato_multicentro', 'boolean', c43_n, c43_v, c43_z, 'ind_possui_consumo', 'boolean', c44_n, c44_v, c44_z, 'dateingest', 'date', c45_n, c45_v, c45_z, 'yearingest', 'string', c46_n, c46_v, c46_z, 'monthingest', 'string', c47_n, c47_v, c47_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`cod_fornecedor` IS NULL) AS c24_n, COUNT_IF(`cod_fornecedor` IS NOT NULL AND LOWER(TRIM(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c24_v, COUNT_IF(TRIM(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$') AS c24_z, COUNT_IF(`nm_fornecedor` IS NULL) AS c25_n, COUNT_IF(`nm_fornecedor` IS NOT NULL AND LOWER(TRIM(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c25_v, COUNT_IF(TRIM(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$') AS c25_z, COUNT_IF(`cod_grupo_compradores` IS NULL) AS c26_n, COUNT_IF(`cod_grupo_compradores` IS NOT NULL AND LOWER(TRIM(`cod_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c26_v, COUNT_IF(TRIM(`cod_grupo_compradores`) RLIKE '^0+([.,]0+)?$') AS c26_z, COUNT_IF(`cod_organizacao_compras` IS NULL) AS c27_n, COUNT_IF(`cod_organizacao_compras` IS NOT NULL AND LOWER(TRIM(`cod_organizacao_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c27_v, COUNT_IF(TRIM(`cod_organizacao_compras`) RLIKE '^0+([.,]0+)?$') AS c27_z, COUNT_IF(`dt_inicio_validade` IS NULL) AS c28_n, CAST(0 AS BIGINT) AS c28_v, CAST(0 AS BIGINT) AS c28_z, COUNT_IF(`dt_fim_validade` IS NULL) AS c29_n, CAST(0 AS BIGINT) AS c29_v, CAST(0 AS BIGINT) AS c29_z, COUNT_IF(`dt_documento_compras` IS NULL) AS c30_n, CAST(0 AS BIGINT) AS c30_v, CAST(0 AS BIGINT) AS c30_z, COUNT_IF(`dt_criacao_cabecalho` IS NULL) AS c31_n, CAST(0 AS BIGINT) AS c31_v, CAST(0 AS BIGINT) AS c31_z, COUNT_IF(`cod_liberacao` IS NULL) AS c32_n, COUNT_IF(`cod_liberacao` IS NOT NULL AND LOWER(TRIM(`cod_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c32_v, COUNT_IF(TRIM(`cod_liberacao`) RLIKE '^0+([.,]0+)?$') AS c32_z, COUNT_IF(`cod_eliminacao_cabecalho` IS NULL) AS c33_n, COUNT_IF(`cod_eliminacao_cabecalho` IS NOT NULL AND LOWER(TRIM(`cod_eliminacao_cabecalho`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c33_v, COUNT_IF(TRIM(`cod_eliminacao_cabecalho`) RLIKE '^0+([.,]0+)?$') AS c33_z, COUNT_IF(`vl_fixado_cabecalho` IS NULL) AS c34_n, CAST(0 AS BIGINT) AS c34_v, COUNT_IF(`vl_fixado_cabecalho` = 0) AS c34_z, COUNT_IF(`cod_centro_custo` IS NULL) AS c35_n, COUNT_IF(`cod_centro_custo` IS NOT NULL AND LOWER(TRIM(`cod_centro_custo`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c35_v, COUNT_IF(TRIM(`cod_centro_custo`) RLIKE '^0+([.,]0+)?$') AS c35_z, COUNT_IF(`num_ordem` IS NULL) AS c36_n, COUNT_IF(`num_ordem` IS NOT NULL AND LOWER(TRIM(`num_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c36_v, COUNT_IF(TRIM(`num_ordem`) RLIKE '^0+([.,]0+)?$') AS c36_z, COUNT_IF(`id_interno_elemento_pep` IS NULL) AS c37_n, COUNT_IF(`id_interno_elemento_pep` IS NOT NULL AND LOWER(TRIM(`id_interno_elemento_pep`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c37_v, COUNT_IF(TRIM(`id_interno_elemento_pep`) RLIKE '^0+([.,]0+)?$') AS c37_z, COUNT_IF(`qtd_classificacoes_contabeis` IS NULL) AS c38_n, CAST(0 AS BIGINT) AS c38_v, COUNT_IF(`qtd_classificacoes_contabeis` = 0) AS c38_z, COUNT_IF(`vlr_consumido` IS NULL) AS c39_n, CAST(0 AS BIGINT) AS c39_v, COUNT_IF(`vlr_consumido` = 0) AS c39_z, COUNT_IF(`qtd_itens_pedido_consumo` IS NULL) AS c40_n, CAST(0 AS BIGINT) AS c40_v, COUNT_IF(`qtd_itens_pedido_consumo` = 0) AS c40_z, COUNT_IF(`ind_servico` IS NULL) AS c41_n, CAST(0 AS BIGINT) AS c41_v, CAST(0 AS BIGINT) AS c41_z, COUNT_IF(`ind_possui_cabecalho` IS NULL) AS c42_n, CAST(0 AS BIGINT) AS c42_v, CAST(0 AS BIGINT) AS c42_z, COUNT_IF(`ind_contrato_multicentro` IS NULL) AS c43_n, CAST(0 AS BIGINT) AS c43_v, CAST(0 AS BIGINT) AS c43_z, COUNT_IF(`ind_possui_consumo` IS NULL) AS c44_n, CAST(0 AS BIGINT) AS c44_v, CAST(0 AS BIGINT) AS c44_z, COUNT_IF(`dateingest` IS NULL) AS c45_n, CAST(0 AS BIGINT) AS c45_v, CAST(0 AS BIGINT) AS c45_z, COUNT_IF(`yearingest` IS NULL) AS c46_n, COUNT_IF(`yearingest` IS NOT NULL AND LOWER(TRIM(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c46_v, COUNT_IF(TRIM(`yearingest`) RLIKE '^0+([.,]0+)?$') AS c46_z, COUNT_IF(`monthingest` IS NULL) AS c47_n, COUNT_IF(`monthingest` IS NOT NULL AND LOWER(TRIM(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c47_v, COUNT_IF(TRIM(`monthingest`) RLIKE '^0+([.,]0+)?$') AS c47_z FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) a), t AS (SELECT COUNT(*) AS total FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) SELECT p.coluna, p.tipo, t.total, p.nulos, p.vazios, p.zeros, t.total - p.nulos - p.vazios AS preenchidos_nao_vazios, CASE WHEN t.total = 0 THEN 'SEM LINHAS' WHEN p.nulos = t.total THEN '100% NULO - CONFERIR COMENTARIO' WHEN p.nulos + p.vazios = t.total THEN '100% SEM VALOR UTIL - CONFERIR' ELSE 'COM DADOS' END AS veredito FROM perf p CROSS JOIN t ORDER BY p.coluna;


## 6.1. Colunas sem fonte

Não há coluna explicitamente classificada como sem fonte no DESCRIBE; não conclui que todas são preenchidas.


In [0]:
SELECT 'Nenhuma coluna explicitamente classificada como sem fonte no DESCRIBE' AS observacao;


## 7. Cardinalidade

Identifica atributos constantes, categorias e possíveis identificadores sem expor valores pessoais.


In [0]:
WITH a AS (SELECT COUNT(*) AS total, COUNT(DISTINCT `cod_fornecedor`) AS d0, COUNT(DISTINCT `num_documento_compras`) AS d1, COUNT(DISTINCT `cod_material`) AS d2, COUNT(DISTINCT `num_item_documento_compras`) AS d3, COUNT(DISTINCT `tp_categoria_documento_compras`) AS d4, COUNT(DISTINCT `tp_documento_compras`) AS d5, COUNT(DISTINCT `cod_centro`) AS d6, COUNT(DISTINCT `cod_deposito`) AS d7, COUNT(DISTINCT `cod_grupo_mercadorias`) AS d8, COUNT(DISTINCT `cod_unidade_medida_pedido`) AS d9 FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) SELECT coluna, distintos, total, CASE WHEN distintos <= 1 THEN 'CONSTANTE OU VAZIO' WHEN distintos <= 20 THEN 'BAIXA CARDINALIDADE' WHEN distintos >= total * 0.9 THEN 'CANDIDATO A IDENTIFICADOR' ELSE 'INTERMEDIARIA' END AS perfil FROM (SELECT total, stack(10, 'cod_fornecedor', d0, 'num_documento_compras', d1, 'cod_material', d2, 'num_item_documento_compras', d3, 'tp_categoria_documento_compras', d4, 'tp_documento_compras', d5, 'cod_centro', d6, 'cod_deposito', d7, 'cod_grupo_mercadorias', d8, 'cod_unidade_medida_pedido', d9) AS (coluna, distintos) FROM a) s ORDER BY distintos DESC;


## 8. Domínio

Mostra até oito valores por campo categórico não pessoal; nomes, identificadores pessoais e endereços ficam fora.


In [0]:
WITH d AS (SELECT 'tp_categoria_documento_compras' AS coluna, CAST(`tp_categoria_documento_compras` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `tp_categoria_documento_compras` UNION ALL SELECT 'tp_documento_compras' AS coluna, CAST(`tp_documento_compras` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `tp_documento_compras` UNION ALL SELECT 'cod_centro' AS coluna, CAST(`cod_centro` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `cod_centro` UNION ALL SELECT 'cod_deposito' AS coluna, CAST(`cod_deposito` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `cod_deposito` UNION ALL SELECT 'cod_grupo_mercadorias' AS coluna, CAST(`cod_grupo_mercadorias` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `cod_grupo_mercadorias` UNION ALL SELECT 'cod_unidade_medida_pedido' AS coluna, CAST(`cod_unidade_medida_pedido` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `cod_unidade_medida_pedido` UNION ALL SELECT 'cod_unidade_medida_basica' AS coluna, CAST(`cod_unidade_medida_basica` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `cod_unidade_medida_basica` UNION ALL SELECT 'cod_unidade_medida_preco_pedido' AS coluna, CAST(`cod_unidade_medida_preco_pedido` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `cod_unidade_medida_preco_pedido`), r AS (SELECT coluna, valor, linhas, ROW_NUMBER() OVER (PARTITION BY coluna ORDER BY linhas DESC, valor) AS pos FROM d) SELECT coluna, valor, linhas FROM r WHERE pos <= 8 ORDER BY coluna, pos;


## 9. Booleanas e indicadores

Mede os estados observados, sem converter indicadores STRING em booleanos por suposição.


In [0]:
WITH a AS (SELECT 'tp_categoria_documento_compras' AS coluna, CAST(`tp_categoria_documento_compras` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `tp_categoria_documento_compras` UNION ALL SELECT 'tp_documento_compras' AS coluna, CAST(`tp_documento_compras` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `tp_documento_compras` UNION ALL SELECT 'tp_categoria_item_compra' AS coluna, CAST(`tp_categoria_item_compra` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `tp_categoria_item_compra` UNION ALL SELECT 'tp_classificacao_contabil' AS coluna, CAST(`tp_classificacao_contabil` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `tp_classificacao_contabil` UNION ALL SELECT 'ind_servico' AS coluna, CAST(`ind_servico` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `ind_servico` UNION ALL SELECT 'ind_possui_cabecalho' AS coluna, CAST(`ind_possui_cabecalho` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `ind_possui_cabecalho` UNION ALL SELECT 'ind_contrato_multicentro' AS coluna, CAST(`ind_contrato_multicentro` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `ind_contrato_multicentro` UNION ALL SELECT 'ind_possui_consumo' AS coluna, CAST(`ind_possui_consumo` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `ind_possui_consumo`) SELECT coluna, valor, linhas FROM a ORDER BY coluna, linhas DESC;


## 10. Perfil numérico

Perfila quantidades e valores. Campos de códigos numéricos não são tratados como montantes.


In [0]:
SELECT * FROM (SELECT stack(8, 'vl_preco_liquido', CAST(n0_min AS DOUBLE), CAST(n0_max AS DOUBLE), CAST(n0_avg AS DOUBLE), CAST(n0_med AS DOUBLE), n0_neg, n0_zero, 'qt_unidade_preco', CAST(n1_min AS DOUBLE), CAST(n1_max AS DOUBLE), CAST(n1_avg AS DOUBLE), CAST(n1_med AS DOUBLE), n1_neg, n1_zero, 'qt_prevista_contrato', CAST(n2_min AS DOUBLE), CAST(n2_max AS DOUBLE), CAST(n2_avg AS DOUBLE), CAST(n2_med AS DOUBLE), n2_neg, n2_zero, 'vl_fixado_item', CAST(n3_min AS DOUBLE), CAST(n3_max AS DOUBLE), CAST(n3_avg AS DOUBLE), CAST(n3_med AS DOUBLE), n3_neg, n3_zero, 'vl_fixado_cabecalho', CAST(n4_min AS DOUBLE), CAST(n4_max AS DOUBLE), CAST(n4_avg AS DOUBLE), CAST(n4_med AS DOUBLE), n4_neg, n4_zero, 'qtd_classificacoes_contabeis', CAST(n5_min AS DOUBLE), CAST(n5_max AS DOUBLE), CAST(n5_avg AS DOUBLE), CAST(n5_med AS DOUBLE), n5_neg, n5_zero, 'vlr_consumido', CAST(n6_min AS DOUBLE), CAST(n6_max AS DOUBLE), CAST(n6_avg AS DOUBLE), CAST(n6_med AS DOUBLE), n6_neg, n6_zero, 'qtd_itens_pedido_consumo', CAST(n7_min AS DOUBLE), CAST(n7_max AS DOUBLE), CAST(n7_avg AS DOUBLE), CAST(n7_med AS DOUBLE), n7_neg, n7_zero) AS (coluna, minimo, maximo, media, mediana_aproximada, negativos, zeros) FROM (SELECT MIN(`vl_preco_liquido`) AS n0_min, MAX(`vl_preco_liquido`) AS n0_max, AVG(`vl_preco_liquido`) AS n0_avg, PERCENTILE_APPROX(`vl_preco_liquido`, 0.5) AS n0_med, COUNT_IF(`vl_preco_liquido` < 0) AS n0_neg, COUNT_IF(`vl_preco_liquido` = 0) AS n0_zero, MIN(`qt_unidade_preco`) AS n1_min, MAX(`qt_unidade_preco`) AS n1_max, AVG(`qt_unidade_preco`) AS n1_avg, PERCENTILE_APPROX(`qt_unidade_preco`, 0.5) AS n1_med, COUNT_IF(`qt_unidade_preco` < 0) AS n1_neg, COUNT_IF(`qt_unidade_preco` = 0) AS n1_zero, MIN(`qt_prevista_contrato`) AS n2_min, MAX(`qt_prevista_contrato`) AS n2_max, AVG(`qt_prevista_contrato`) AS n2_avg, PERCENTILE_APPROX(`qt_prevista_contrato`, 0.5) AS n2_med, COUNT_IF(`qt_prevista_contrato` < 0) AS n2_neg, COUNT_IF(`qt_prevista_contrato` = 0) AS n2_zero, MIN(`vl_fixado_item`) AS n3_min, MAX(`vl_fixado_item`) AS n3_max, AVG(`vl_fixado_item`) AS n3_avg, PERCENTILE_APPROX(`vl_fixado_item`, 0.5) AS n3_med, COUNT_IF(`vl_fixado_item` < 0) AS n3_neg, COUNT_IF(`vl_fixado_item` = 0) AS n3_zero, MIN(`vl_fixado_cabecalho`) AS n4_min, MAX(`vl_fixado_cabecalho`) AS n4_max, AVG(`vl_fixado_cabecalho`) AS n4_avg, PERCENTILE_APPROX(`vl_fixado_cabecalho`, 0.5) AS n4_med, COUNT_IF(`vl_fixado_cabecalho` < 0) AS n4_neg, COUNT_IF(`vl_fixado_cabecalho` = 0) AS n4_zero, MIN(`qtd_classificacoes_contabeis`) AS n5_min, MAX(`qtd_classificacoes_contabeis`) AS n5_max, AVG(`qtd_classificacoes_contabeis`) AS n5_avg, PERCENTILE_APPROX(`qtd_classificacoes_contabeis`, 0.5) AS n5_med, COUNT_IF(`qtd_classificacoes_contabeis` < 0) AS n5_neg, COUNT_IF(`qtd_classificacoes_contabeis` = 0) AS n5_zero, MIN(`vlr_consumido`) AS n6_min, MAX(`vlr_consumido`) AS n6_max, AVG(`vlr_consumido`) AS n6_avg, PERCENTILE_APPROX(`vlr_consumido`, 0.5) AS n6_med, COUNT_IF(`vlr_consumido` < 0) AS n6_neg, COUNT_IF(`vlr_consumido` = 0) AS n6_zero, MIN(`qtd_itens_pedido_consumo`) AS n7_min, MAX(`qtd_itens_pedido_consumo`) AS n7_max, AVG(`qtd_itens_pedido_consumo`) AS n7_avg, PERCENTILE_APPROX(`qtd_itens_pedido_consumo`, 0.5) AS n7_med, COUNT_IF(`qtd_itens_pedido_consumo` < 0) AS n7_neg, COUNT_IF(`qtd_itens_pedido_consumo` = 0) AS n7_zero FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) a) n ORDER BY coluna;


## 11. Totais para conciliação

Soma separadamente cada quantidade e valor. Totais DOUBLE são aproximados; comparar moedas e granularidade antes de conciliar.


In [0]:
SELECT coluna, total FROM (SELECT stack(8, 'vl_preco_liquido', CAST(s0 AS DOUBLE), 'qt_unidade_preco', CAST(s1 AS DOUBLE), 'qt_prevista_contrato', CAST(s2 AS DOUBLE), 'vl_fixado_item', CAST(s3 AS DOUBLE), 'vl_fixado_cabecalho', CAST(s4 AS DOUBLE), 'qtd_classificacoes_contabeis', CAST(s5 AS DOUBLE), 'vlr_consumido', CAST(s6 AS DOUBLE), 'qtd_itens_pedido_consumo', CAST(s7 AS DOUBLE)) AS (coluna, total) FROM (SELECT SUM(`vl_preco_liquido`) AS s0, SUM(`qt_unidade_preco`) AS s1, SUM(`qt_prevista_contrato`) AS s2, SUM(`vl_fixado_item`) AS s3, SUM(`vl_fixado_cabecalho`) AS s4, SUM(`qtd_classificacoes_contabeis`) AS s5, SUM(`vlr_consumido`) AS s6, SUM(`qtd_itens_pedido_consumo`) AS s7 FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) a) s ORDER BY coluna;


## 12. Datas

Verifica sentinelas e formatos; mínimos de campos STRING são literais, não datas cronológicas.


In [0]:
SELECT * FROM (SELECT 'dt_inicio_validade' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_inicio_validade` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_inicio_validade` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_inicio_validade`) AS STRING) AS menor_literal, CAST(MAX(`dt_inicio_validade`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` UNION ALL SELECT 'dt_fim_validade' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_fim_validade` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_fim_validade` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_fim_validade`) AS STRING) AS menor_literal, CAST(MAX(`dt_fim_validade`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` UNION ALL SELECT 'dt_documento_compras' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_documento_compras` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_documento_compras` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_documento_compras`) AS STRING) AS menor_literal, CAST(MAX(`dt_documento_compras`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` UNION ALL SELECT 'dt_criacao_cabecalho' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_criacao_cabecalho` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_criacao_cabecalho` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_criacao_cabecalho`) AS STRING) AS menor_literal, CAST(MAX(`dt_criacao_cabecalho`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` UNION ALL SELECT 'dateingest' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dateingest` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dateingest` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dateingest`) AS STRING) AS menor_literal, CAST(MAX(`dateingest`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) d ORDER BY coluna;


## 13. Códigos

Detecta zeros à esquerda e possíveis colisões ao normalizar identificadores não pessoais. Não normalizar sem avaliar a semântica.


In [0]:
SELECT *, codigos_numericos_distintos - codigos_normalizados_distintos AS colisoes_potenciais FROM (SELECT 'num_documento_compras' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_documento_compras`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_documento_compras`))) AS tamanho_max, COUNT_IF(TRIM(`num_documento_compras`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_documento_compras`) RLIKE '^[0-9]+$' THEN TRIM(`num_documento_compras`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_documento_compras`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_documento_compras`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` UNION ALL SELECT 'num_item_documento_compras' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_item_documento_compras`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_item_documento_compras`))) AS tamanho_max, COUNT_IF(TRIM(`num_item_documento_compras`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_item_documento_compras`) RLIKE '^[0-9]+$' THEN TRIM(`num_item_documento_compras`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_item_documento_compras`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_item_documento_compras`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` UNION ALL SELECT 'cod_material' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_material`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_material`))) AS tamanho_max, COUNT_IF(TRIM(`cod_material`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_material`) RLIKE '^[0-9]+$' THEN TRIM(`cod_material`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_material`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_material`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` UNION ALL SELECT 'cod_centro' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_centro`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_centro`))) AS tamanho_max, COUNT_IF(TRIM(`cod_centro`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_centro`) RLIKE '^[0-9]+$' THEN TRIM(`cod_centro`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_centro`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_centro`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` UNION ALL SELECT 'cod_deposito' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_deposito`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_deposito`))) AS tamanho_max, COUNT_IF(TRIM(`cod_deposito`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_deposito`) RLIKE '^[0-9]+$' THEN TRIM(`cod_deposito`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_deposito`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_deposito`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` UNION ALL SELECT 'cod_grupo_mercadorias' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_grupo_mercadorias`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_grupo_mercadorias`))) AS tamanho_max, COUNT_IF(TRIM(`cod_grupo_mercadorias`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_grupo_mercadorias`) RLIKE '^[0-9]+$' THEN TRIM(`cod_grupo_mercadorias`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_grupo_mercadorias`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_grupo_mercadorias`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` UNION ALL SELECT 'cod_unidade_medida_pedido' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_unidade_medida_pedido`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_unidade_medida_pedido`))) AS tamanho_max, COUNT_IF(TRIM(`cod_unidade_medida_pedido`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_unidade_medida_pedido`) RLIKE '^[0-9]+$' THEN TRIM(`cod_unidade_medida_pedido`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_unidade_medida_pedido`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_unidade_medida_pedido`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` UNION ALL SELECT 'cod_unidade_medida_basica' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_unidade_medida_basica`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_unidade_medida_basica`))) AS tamanho_max, COUNT_IF(TRIM(`cod_unidade_medida_basica`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_unidade_medida_basica`) RLIKE '^[0-9]+$' THEN TRIM(`cod_unidade_medida_basica`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_unidade_medida_basica`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_unidade_medida_basica`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` UNION ALL SELECT 'cod_unidade_medida_preco_pedido' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_unidade_medida_preco_pedido`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_unidade_medida_preco_pedido`))) AS tamanho_max, COUNT_IF(TRIM(`cod_unidade_medida_preco_pedido`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_unidade_medida_preco_pedido`) RLIKE '^[0-9]+$' THEN TRIM(`cod_unidade_medida_preco_pedido`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_unidade_medida_preco_pedido`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_unidade_medida_preco_pedido`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` UNION ALL SELECT 'cod_iva' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_iva`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_iva`))) AS tamanho_max, COUNT_IF(TRIM(`cod_iva`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_iva`) RLIKE '^[0-9]+$' THEN TRIM(`cod_iva`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_iva`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_iva`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) c ORDER BY coluna;


## 14. Checksum

Conta linhas 100% idênticas por hash da linha inteira; não retorna conteúdo nem hashes de dados pessoais.


In [0]:
WITH h AS (SELECT md5(to_json(struct(t.`num_documento_compras`, t.`num_item_documento_compras`, t.`tp_categoria_documento_compras`, t.`tp_documento_compras`, t.`cod_material`, t.`desc_texto_breve_item`, t.`cod_centro`, t.`cod_deposito`, t.`cod_grupo_mercadorias`, t.`cod_unidade_medida_pedido`, t.`cod_unidade_medida_basica`, t.`cod_unidade_medida_preco_pedido`, t.`vl_preco_liquido`, t.`qt_unidade_preco`, t.`qt_prevista_contrato`, t.`vl_fixado_item`, t.`cod_iva`, t.`cod_domicilio_fiscal`, t.`cod_registro_info_compras`, t.`tp_categoria_item_compra`, t.`tp_classificacao_contabil`, t.`cod_eliminacao_item`, t.`cod_empresa`, t.`cod_moeda`, t.`cod_fornecedor`, t.`nm_fornecedor`, t.`cod_grupo_compradores`, t.`cod_organizacao_compras`, t.`dt_inicio_validade`, t.`dt_fim_validade`, t.`dt_documento_compras`, t.`dt_criacao_cabecalho`, t.`cod_liberacao`, t.`cod_eliminacao_cabecalho`, t.`vl_fixado_cabecalho`, t.`cod_centro_custo`, t.`num_ordem`, t.`id_interno_elemento_pep`, t.`qtd_classificacoes_contabeis`, t.`vlr_consumido`, t.`qtd_itens_pedido_consumo`, t.`ind_servico`, t.`ind_possui_cabecalho`, t.`ind_contrato_multicentro`, t.`ind_possui_consumo`, t.`dateingest`, t.`yearingest`, t.`monthingest`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` t), a AS (SELECT hash_linha, COUNT(*) AS linhas FROM h GROUP BY hash_linha) SELECT COALESCE(SUM(linhas),0) AS linhas, COUNT(*) AS linhas_distintas, COALESCE(SUM(linhas - 1),0) AS repeticoes_identicas FROM a;


## 15. Amostra

Mostra 20 registros com colunas não pessoais; campos potencialmente pessoais são omitidos ou hash de 8 caracteres.


In [0]:
SELECT `num_documento_compras`, `num_item_documento_compras`, `tp_categoria_documento_compras`, `tp_documento_compras`, `cod_material`, `desc_texto_breve_item`, `cod_centro`, `cod_deposito`, `cod_grupo_mercadorias`, `cod_unidade_medida_pedido`, `cod_unidade_medida_basica`, `cod_unidade_medida_preco_pedido`, `vl_preco_liquido`, `qt_unidade_preco`, `qt_prevista_contrato`, `vl_fixado_item`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` LIMIT 20;


## 16. Distribuição interna

Compara o mix de categorias e centro, quando houver, sem expor pessoas.


In [0]:
SELECT `cod_centro`, `tp_categoria_documento_compras`, `tp_documento_compras`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `cod_centro`, `tp_categoria_documento_compras`, `tp_documento_compras` ORDER BY linhas DESC LIMIT 100;


## 17. Freshness

Mede datas de carga e partições; não confundir carimbo de ingestão com evento de negócio.


In [0]:
SELECT COUNT(*) AS linhas, MIN(`dateingest`) AS `min_dateingest`, MAX(`dateingest`) AS `max_dateingest`, COUNT(DISTINCT `dateingest`) AS `distintos_dateingest`, MIN(`yearingest`) AS `min_yearingest`, MAX(`yearingest`) AS `max_yearingest`, COUNT(DISTINCT `yearingest`) AS `distintos_yearingest`, MIN(`monthingest`) AS `min_monthingest`, MAX(`monthingest`) AS `max_monthingest`, COUNT(DISTINCT `monthingest`) AS `distintos_monthingest` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`;


## 18. Análises específicas da transação

Testes derivados dos comentários do DESCRIBE; as hipóteses dependem dos resultados medidos.


### 18.1. Contrato, vigência, consumo e classificações


In [0]:
SELECT `tp_categoria_documento_compras`, COUNT(*) AS itens, COUNT_IF(`cod_centro` IS NULL OR TRIM(`cod_centro`) = '') AS sem_centro, COUNT_IF(`dt_inicio_validade` IS NOT NULL AND `dt_fim_validade` IS NOT NULL AND `dt_fim_validade` < `dt_inicio_validade`) AS vigencia_invertida, COUNT_IF(`qtd_classificacoes_contabeis` > 1) AS classificacao_multipla, COUNT_IF(`ind_possui_cabecalho` = FALSE) AS sem_cabecalho, COUNT_IF(`vlr_consumido` IS NULL) AS consumo_nulo FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `tp_categoria_documento_compras` ORDER BY itens DESC;


## 19. Amostra ampliada

Amostra de até 500 registros sem campos pessoais literais; não equivale à extração completa do cenário.


In [0]:
SELECT `num_documento_compras`, `num_item_documento_compras`, `tp_categoria_documento_compras`, `tp_documento_compras`, `cod_material`, `desc_texto_breve_item`, `cod_centro`, `cod_deposito`, `cod_grupo_mercadorias`, `cod_unidade_medida_pedido`, `cod_unidade_medida_basica`, `cod_unidade_medida_preco_pedido`, `vl_preco_liquido`, `qt_unidade_preco`, `qt_prevista_contrato`, `vl_fixado_item`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` LIMIT 500;


## 20. Resumo

Copie este bloco junto com as seções 3, 5, 6 e 18 para validar chave real e filtros antes dos cenários.


In [0]:
WITH d AS (SELECT `num_documento_compras`, `num_item_documento_compras`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra` GROUP BY `num_documento_compras`, `num_item_documento_compras`), v AS (SELECT COUNT(*) AS linhas, COUNT(DISTINCT `cod_centro`) AS centros_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_me3n_contratos_compra`) SELECT 'ME3N' AS transacao, 'tbl_ds_pro_me3n_contratos_compra' AS objeto, v.linhas, (SELECT COUNT(*) FROM d) AS chaves_candidatas_distintas, (SELECT COUNT(*) FROM d WHERE linhas > 1) AS chaves_candidatas_repetidas, v.centros_distintos, 'Chave candidata: num_documento_compras + num_item_documento_compras' AS nota_chave FROM v;


## Checklist para comparação com SAP

- [ ] Confirmar comentário do objeto e filtros aplicados na tela SAP.
- [ ] Registrar instante de execução e mesmo recorte temporal da extração SAP.
- [ ] Validar chave real, duplicatas idênticas e versões distintas.
- [ ] Conferir centros 4128 e 4014 somente quando há campo de centro; caso contrário, definir correspondência de negócio antes de comparar.
- [ ] Verificar nulos sem declaração de sem fonte; não classificar lacuna documentada como defeito.
- [ ] Normalizar códigos com zeros à esquerda e datas; validar separador decimal, moeda e tolerância de 0,005 para DOUBLE.
- [ ] Não extrair nem divulgar valores pessoais literais.
- [ ] Executar cenários somente após receber os resultados deste notebook.
